# Olist E-Commerce — Data Exploration 
## Goal: Understand all 9 tables, their shapes, data types, nulls, and relationships

In [2]:
import pandas as pd
import numpy as np
import os

DATA_PATH = '../data/'

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Setup complete')


Setup complete


In [4]:
orders = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
customers = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
order_items = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
payments = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
products = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')
geo = pd.read_csv(DATA_PATH + 'olist_geolocation_dataset.csv')
translation = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

# Quick size check — all tables at a glance
tables = { 'orders': orders,
           'customers': customers,
           'order_items': order_items,
           'payments': payments,
           'reviews': reviews,
           'products': products,
           'sellers': sellers,
           'geo': geo,
           'translation': translation }

print(f"{'Table':<20} {'Rows':>8} {'Columns':>8}")
print("-" * 38)
for name, df in tables.items():
    print(f"{name:<20} {len(df):>8,} {len(df.columns):>8}")

Table                    Rows  Columns
--------------------------------------
orders                 99,441        8
customers              99,441        5
order_items           112,650        7
payments              103,886        5
reviews                99,224        7
products               32,951        9
sellers                 3,095        4
geo                  1,000,163        5
translation                71        2


In [8]:
def inspect_table(df, name):
    """Prints a comprehensive diagnostic report for a given Pandas DataFrame."""
    # 1. Print a prominent header for visual separation
    print("=" * 60)
    print(f"TABLE: {name.upper()}")
    print("=" * 60)
    
    # 2. Print dimensions (Rows x Columns)
    print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # 3. Print data types of all columns
    print(f"\nColumns & Data Types:")
    print(df.dtypes.to_string())
    
    # 4. Calculate and isolate missing/null values
    print(f"\nNull Values:")
    nulls = df.isnull().sum()
    null_pct = (nulls / len(df) * 100).round(1)
    
    # Create a temporary DataFrame to hold the counts and percentages
    null_report = pd.DataFrame({'null_count': nulls, 'null_%': null_pct})
    
    # Filter out columns that have 0 missing values
    null_report = null_report[null_report['null_count'] > 0]
    
    # Check if the filtered report is empty or contains rows
    if len(null_report) == 0:
        print("No null values found")
    else:
        print(null_report.to_string())
        
    # 5. Print a quick preview of the raw data
    print(f"\nSample rows (first 3):")
    print(df.head(3).to_string())
    print()

In [9]:
inspect_table(orders, 'orders')

TABLE: ORDERS

  Shape: 99,441 rows × 8 columns

Columns & Data Types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

Null Values:
                               null_count  null_%
order_approved_at                     160    0.20
order_delivered_carrier_date         1783    1.80
order_delivered_customer_date        2965    3.00

Sample rows (first 3):
                           order_id                       customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-1

In [10]:
inspect_table(customers, 'customers')

TABLE: CUSTOMERS

  Shape: 99,441 rows × 5 columns

Columns & Data Types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str

Null Values:
No null values found

Sample rows (first 3):
                        customer_id                customer_unique_id  customer_zip_code_prefix          customer_city customer_state
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0                     14409                 franca             SP
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3                      9790  sao bernardo do campo             SP
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e                      1151              sao paulo             SP



In [11]:
inspect_table(order_items, 'order_items')

TABLE: ORDER_ITEMS

  Shape: 112,650 rows × 7 columns

Columns & Data Types:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64

Null Values:
No null values found

Sample rows (first 3):
                           order_id  order_item_id                        product_id                         seller_id  shipping_limit_date  price  freight_value
0  00010242fe8c5a6d1ba2dd792cb16214              1  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202  2017-09-19 09:45:35  58.90          13.29
1  00018f77f2f0320c557190d7a144bdd3              1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36  2017-05-03 11:05:13 239.90          19.93
2  000229ec398224ef6ca0657da4fc703e              1  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d  2018-01-18 14:48:30 199.00          17.87

In [12]:
inspect_table(payments, 'payments')

TABLE: PAYMENTS

  Shape: 103,886 rows × 5 columns

Columns & Data Types:
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64

Null Values:
No null values found

Sample rows (first 3):
                           order_id  payment_sequential payment_type  payment_installments  payment_value
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card                     8          99.33
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card                     1          24.39
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card                     1          65.71



In [13]:
inspect_table(reviews, 'reviews')

TABLE: REVIEWS

  Shape: 99,224 rows × 7 columns

Columns & Data Types:
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str

Null Values:
                        null_count  null_%
review_comment_title         87656   88.30
review_comment_message       58247   58.70

Sample rows (first 3):
                          review_id                          order_id  review_score review_comment_title review_comment_message review_creation_date review_answer_timestamp
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb             4                  NaN                    NaN  2018-01-18 00:00:00     2018-01-18 21:46:59
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33             5                  NaN                    NaN  2018-03-10 00:00:00     2018-03-11 03:05:13
2  228ce550

In [14]:
inspect_table(products, 'products')

TABLE: PRODUCTS

  Shape: 32,951 rows × 9 columns

Columns & Data Types:
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64

Null Values:
                            null_count  null_%
product_category_name              610    1.90
product_name_lenght                610    1.90
product_description_lenght         610    1.90
product_photos_qty                 610    1.90
product_weight_g                     2    0.00
product_length_cm                    2    0.00
product_height_cm                    2    0.00
product_width_cm                     2    0.00

Sample rows (first 3):
                         product_id product_category_name  product_name_lenght  product_description_lenght  product_photos_q

In [15]:
inspect_table(sellers, 'sellers')

TABLE: SELLERS

  Shape: 3,095 rows × 4 columns

Columns & Data Types:
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str

Null Values:
No null values found

Sample rows (first 3):
                          seller_id  seller_zip_code_prefix     seller_city seller_state
0  3442f8959a84dea7ee197c632cb2df15                   13023        campinas           SP
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844      mogi guacu           SP
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031  rio de janeiro           RJ



In [16]:
inspect_table(translation, 'translation')

TABLE: TRANSLATION

  Shape: 71 rows × 2 columns

Columns & Data Types:
product_category_name            str
product_category_name_english    str

Null Values:
No null values found

Sample rows (first 3):
    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto



In [17]:
# Master null summary across all tables
print(f"{'Table':<20} {'Column':<35} {'Nulls':>8} {'%':>6}")
print("-" * 72)

for name, df in tables.items():
    nulls = df.isnull().sum()
    
    # Filter and loop only through columns that actually have missing values
    for col, count in nulls[nulls > 0].items():
        pct = (count / len(df)) * 100
        print(f"{name:<20} {col:<35} {count:>8,} {pct:>5.1f}%")

Table                Column                                 Nulls      %
------------------------------------------------------------------------
orders               order_approved_at                        160   0.2%
orders               order_delivered_carrier_date           1,783   1.8%
orders               order_delivered_customer_date          2,965   3.0%
reviews              review_comment_title                  87,656  88.3%
reviews              review_comment_message                58,247  58.7%
products             product_category_name                    610   1.9%
products             product_name_lenght                      610   1.9%
products             product_description_lenght               610   1.9%
products             product_photos_qty                       610   1.9%
products             product_weight_g                           2   0.0%
products             product_length_cm                          2   0.0%
products             product_height_cm             

In [18]:
# Unique value counts for all join keys
key_checks = [
    ('orders', 'order_id', 'primary key?'),
    ('orders', 'customer_id', 'joins → customers'),
    ('customers', 'customer_id', 'primary key?'),
    ('customers', 'customer_unique_id', 'real unique customer'),
    ('order_items', 'order_id', 'joins → orders (many per order)'),
    ('order_items', 'product_id', 'joins → products'),
    ('order_items', 'seller_id', 'joins → sellers'),
    ('payments', 'order_id', 'joins → orders'),
    ('reviews', 'order_id', 'joins → orders'),
    ('products', 'product_id', 'primary key?'),
    ('sellers', 'seller_id', 'primary key?'),
]

print(f"{'Table':<15} {'Column':<25} {'Unique':>8} {'Total':>8}  Note")
print("-" * 80)

for tbl, col, note in key_checks:
    df = tables[tbl]
    unique_count = df[col].nunique()
    total_count = len(df)
    print(f"{tbl:<15} {col:<25} {unique_count:>8,} {total_count:>8,}  {note}")

Table           Column                      Unique    Total  Note
--------------------------------------------------------------------------------
orders          order_id                    99,441   99,441  primary key?
orders          customer_id                 99,441   99,441  joins → customers
customers       customer_id                 99,441   99,441  primary key?
customers       customer_unique_id          96,096   99,441  real unique customer
order_items     order_id                    98,666  112,650  joins → orders (many per order)
order_items     product_id                  32,951  112,650  joins → products
order_items     seller_id                    3,095  112,650  joins → sellers
payments        order_id                    99,440  103,886  joins → orders
reviews         order_id                    98,673   99,224  joins → orders
products        product_id                  32,951   32,951  primary key?
sellers         seller_id                    3,095    3,095  primary k

In [19]:
# Test: join orders + customers
test1 = orders.merge(customers, on='customer_id', how='left')
print(f"orders rows: {len(orders):,}")
print(f"after join customers: {len(test1):,} ← should still be 99,441")

# Test: join orders + order_items (expect MORE rows — one-to-many)
test2 = orders.merge(order_items, on='order_id', how='left')
print(f"\nafter join order_items: {len(test2):,} ← expect ~112,650 (multiple items per order)")

# Check for any orders with NO items (should be ~0 or very small)
missing_items = test2[test2['product_id'].isnull()]
print(f"Orders with no items: {len(missing_items):,}")

orders rows: 99,441
after join customers: 99,441 ← should still be 99,441

after join order_items: 113,425 ← expect ~112,650 (multiple items per order)
Orders with no items: 775


In [23]:
print(orders.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [24]:
# 1. Convert date columns from strings to datetime objects
date_cols = [
    'order_purchase_timestamp', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# 2. Translate product category names to English immediately
products = products.merge(translation, on='product_category_name', how='left')

# Drop the old Portuguese name column and clean up the English column name
products = products.drop(columns=['product_category_name'])
products = products.rename(columns={'product_category_name_english': 'product_category'})

# 3. Create a clean master dataframe for customer-level metrics
customer_orders = orders.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']], 
    on='customer_id', 
    how='left'
)

# Quick validation printout
print("✓ Date columns converted to datetime objects.")
print(f"✓ Products translated. 'product_category' now contains English names.")
print(f"✓ Base customer_orders compiled. Shape: {customer_orders.shape[0]:,} rows.")

✓ Date columns converted to datetime objects.
✓ Products translated. 'product_category' now contains English names.
✓ Base customer_orders compiled. Shape: 99,441 rows.


In [25]:
findings = """
KEY FINDINGS
==================
1. SHAPE:
   - 99,441 orders from 96,096 unique customers (some customers ordered multiple times)
   - 112,650 order items across those orders (avg ~1.1 items per order)
   - 32,951 unique products, 3,095 sellers

2. KEY NULLS TO HANDLE:
   - order_delivered_customer_date: 2,965 nulls (3%) — orders not yet delivered
   - product_category_name: 610 nulls — fix using translation table
   - review comments: 58–88% null — optional fields, we will ignore them

3. CRITICAL DISCOVERY:
   - customer_id ≠ customer_unique_id
   - 99,441 customer_ids but only 96,096 unique customers
   - Must use customer_unique_id for churn analysis

4. DATA QUALITY ISSUES TO FIX ON DAY 3:
   - All 5 date columns in orders stored as strings (object dtype)
   - product_name_lenght column has a typo in the source data (known issue)
   - Some products have no category name — use translation CSV to fill
"""

print(findings)


KEY FINDINGS
1. SHAPE:
   - 99,441 orders from 96,096 unique customers (some customers ordered multiple times)
   - 112,650 order items across those orders (avg ~1.1 items per order)
   - 32,951 unique products, 3,095 sellers

2. KEY NULLS TO HANDLE:
   - order_delivered_customer_date: 2,965 nulls (3%) — orders not yet delivered
   - product_category_name: 610 nulls — fix using translation table
   - review comments: 58–88% null — optional fields, we will ignore them

3. CRITICAL DISCOVERY:
   - customer_id ≠ customer_unique_id
   - 99,441 customer_ids but only 96,096 unique customers
   - Must use customer_unique_id for churn analysis

4. DATA QUALITY ISSUES TO FIX ON DAY 3:
   - All 5 date columns in orders stored as strings (object dtype)
   - product_name_lenght column has a typo in the source data (known issue)
   - Some products have no category name — use translation CSV to fill



In [1]:
!pip install pandas numpy

In [6]:
import pandas as pd
import numpy as np

DATA_PATH = '../data/'

orders = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
customers = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')

In [7]:
# List of all date columns in your orders table
date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

# Loop through and convert them
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

In [8]:
# Check what statuses exist
print(orders['order_status'].value_counts())

# Flag cancelled orders (1 for cancelled, 0 for active)
orders['is_cancelled'] = np.where(orders['order_status'] == 'canceled', 1, 0)

# Alternatively, if you just want to drop them entirely:
# orders = orders[orders['order_status'] != 'canceled']

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [9]:
# Create a flag for undelivered orders
orders['is_delivered'] = orders['order_delivered_customer_date'].notna().astype(int)

# Optional: Fill missing delivery dates with a placeholder or leave as NaT 
# Leaving them as NaT (Null) is usually preferred for accurate date math.

In [10]:
# Calculate the exact time difference
time_difference = orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']

# Convert that time difference into a decimal number of days
orders['delivery_delay_days'] = time_difference.dt.total_seconds() / (24 * 3600)

# Round it to 2 decimal places for cleanliness
orders['delivery_delay_days'] = orders['delivery_delay_days'].round(2)

In [11]:
# Merge orders and customers (Inner join keeps only orders with valid customer info)
cleaned_master = pd.merge(orders, customers, on='customer_id', how='inner')

In [12]:
# See the first 5 rows and all the newly combined columns
cleaned_master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_cancelled,is_delivered,delivery_delay_days,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,1,-7.11,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,1,-5.36,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,1,-17.25,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,1,-12.98,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,1,-9.24,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [13]:
# Check how many rows and columns you ended up with
print(cleaned_master.shape)

(99441, 15)


In [14]:
# Save to a new CSV file without the messy index column
cleaned_master.to_csv('clean_orders.csv', index=False)
print("Data cleaning complete! 'clean_orders.csv' has been created.")

Data cleaning complete! 'clean_orders.csv' has been created.
